# Clusterição de eixos

Esse notebook contém os passos pós-detecção de pneus pelo YOLOv8 para o reconhecimento de eixos de um caminhão.

## 1.0 Introdução

### 1.1 Motivação

A detecção automática de eixos de caminhões pode ser útil para realizar a fiscalização contra ilegidades em trâsnito em tempo real, como excesso de carga ou configurações de eixos não permitdas. Comumente, abordagens focadas nessa tarefa focam no desenvolvimento de modelos de aprendizagem de máquina para inferir o posicionamento dos eixos e classificá-los. Entretanto, essas abordagens podem exigir a coleta e rotulação de um grande volume de dados. Como alternativa, a equipe propôs uma metodologia analítica para estimar a posição e classe dos eixos em uma imagem de caminhão que contenha informações de localização e tamanho aparente dos pneus. Essa estratégia busca reduzir a necessidade de se construir um modelo adicional para a realização da tarefa.

### 1.2 Objetivo

Este trabalho objetiva demonstrar a possibilidade de reconhecer e classificar eixos de caminhões por tipo a partir de uma imagem contendo a localização dos pneus além suas larguras e alturas aparentes. Os eixos de um caminhão podem possuir uma ou mais pneus e são classificados, nesse trabalho, de acordo com essa quantidade:

1. **Eixo simples**: eixo formado por uma roda;
2. **Eixo duplo**: eixo formado por duas rodas;
3. **Eixo triplo**: eixo formado por três rodas.

Classificações com mais pneus seguem o mesmo princípio. Durante esta etapa, não é verificado a configuração dos eixos nos modelos tandem ou tridem. Além disso, a rodagem desses eixos é analisado.

## 2.0 Metodologia

### 2.1 Materiais

A equipe desenvolveu um método analítico para estimar a posição dos eixos em uma imagem com um caminhão. O algoritmo foi implementado na linguagem Python e testado usando detecção de pneus geradas pelo modelo do YOLOv8 desenvolvido pela equipe.

### 2.2 Descrição do método

O método desenvolvido para estimar a posição dos eixos se baseia na agrupação de pneus com distâncias próximas. Para isso, são analisadas as distâncias entre os pontos centrais dos pneus, comparando-as dois-a-dois para inferir se ambos pertencem ao mesmo eixo. Para realizar a inferência, o método se utiliza da largura aparente do pneu na imagem para determinar a distância máxima entre o centro dos pneus Esse limite leva em consideração o ângulo do caminhão na imagem.

### 2.3 Formato de entrada

O conjunto de dados de entrada precisa possuir as informações de localização e tamanho aparente dos pneus em uma imagem normalizada. A posição de cada pneu é representada pelas coordenadas do seu ponto central (x, y). Além disso, a largura aparente (w) e altura aparente (h) de cada pneu devem ser especificadas.

O formato proposto é, portanto, da seguinte forma:

$$
P =
\begin{bmatrix}
(x_1, y_1, w_1, h_1) \\
(x_2, y_2, w_2, h_2) \\
\vdots \\
(x_n, y_n, w_n, h_n)
\end{bmatrix}
$$

### 2.4 Saída

A saída do método consiste em um vetor que representa os eixos identificados e em uma lista contendo as posições desses eixos.

Cada posição do vetor de eixos corresponde a um pneu detectado na imagem. O valor armazenado nessa posição indica o identificador do eixo ao qual o pneu pertence. Para manter consistência, os pneus são ordenados do pneu mais à esquerda ao mais à direita.

$$
\text{eixos} = (e_1, e_2, \dots, e_n)
$$

A lista de posições dos eixos armazena a região da imagem correspondente a cada eixo identificado. Cada elemento da lista define uma bounding box delimitada pelos pontos que representam, respectivamente, o canto inferior esquerdo e o canto superior direito da região que contém os pneus associados ao eixo.

$$
\text{eBboxes} =
\begin{bmatrix}
(x_{1,\min}, y_{1,\min}) & (x_{1,\max}, y_{1,\max}) \\
(x_{2,\min}, y_{2,\min}) & (x_{2,\max}, y_{2,\max}) \\
\vdots & \vdots \\
(x_{n,\min}, y_{n,\min}) & (x_{n,\max}, y_{n,\max})
\end{bmatrix}
$$

A saída completa proposta é, portanto, no seguinte formato:

$$
S = (\text{eixos},\ \text{eBboxes})
$$

## 3.0 Algoritmo elaborado

### 3.1 Premissas

Para que a metodologia, no estado atual, seja capaz de inferir corretamente a classe e a posição dos eixos em uma imagem de caminhão com pneus detectados, algumas premissas devem ser atendidas:

1. A imagem fornecida pelo modelo YOLOv8 da equipe contém apenas pneus de um único caminhão;
2. Todas as rodas apresentam tamanhos aparentes semelhantes;
3. A distância real entre os eixos do caminhão é, no mínimo, igual à largura de um pneu.

A violação de qualquer uma dessas premissas compromete o funcionamento do método, que não é capaz de lidar com pneus de caminhões diferentes ou com pneus que estejam significativamente rotacionados em relação à maioria.

### 3.2 Modelagem do problema

Os pneus devem ser ordenados do mais a esquerda ao mais a direita. Para cada pneu, até o penúltimo, o algortimo analisa a distância euclidiana do ponto central do pneu até o próximo.

$$
d_i = \sqrt{(x_{i+1} - x_i)^2 + (y_{i+1} - y_i)^2}, \quad i = 1, \dots, n-1
$$

A largura aparente do pneu em análise é utilizada para estimar a distância máxima até o pneu seguinte. Essa distância é tratada como a hipotenusa de um triângulo retângulo, cuja base corresponde à largura aparente do pneu acrescida de um grau de permissividade, que leva em consideração o tamanho aparente do pneu.

$$
\text{permissividade} = \frac{w_i}{h_i}, \qquad
d_{\max} = w_i + (\text{permissividade} \cdot w_i)
$$

O método assume que, em imagens de caminhões fortemente rotacionados em relação à câmera, a distância entre os centros dos pneus aparenta ser menor. Essa rotação também reduz o tamanho aparente dos pneus na imagem. Consequentemente, o fator de permissividade, baseado na largura aparente dos pneus, diminui proporcionalmente, ajustando automaticamente a distância máxima considerada entre pneus do mesmo eixo.

Por fim, supõe-se que a distância euclidiana entre os pontos centrais dos dois pneus não deve exceder a distância máxima calculada para que ambos os pneus pertençam ao mesmo eixo.

As bounding boxes podem ser obtidas por meio de um algoritmo interpretador que, para cada posição do vetor de eixos, relaciona o pneu na posição lida com o valor de eixo indicado para construir dois pontos que representam o canto inferior esquerdo e um canto superior direito de uma bounding box.

### 3.3 Pseudocódigo do algortimo detector

Algortimo de detecção de eixos
Entrada: Lista de pneus P = [(x1,y1,w1,h1), ...]

```
1. Ordenar labels de rodas
2. Inicializar uma lista de eixos com 0's e último_atual como 0
3. Para cada um dos labels, até o penúltimo:
    3.1 Calcule a permissividade
    3.2 Calcule a distância até o próximo pneu
    3.3 Calcule a área da base do triângulo retângulo
    3.4 Calcule a distância máxima
    3.5 eixo[i] = eixo_atual
    3.6 Se a distância até o próximo pneu for maior que a distância máxima:
        3.6.1 eixo_atual++
    3.7 eixo[i + 1] = eixo_atual
4. Retorne eixos
```